<a href="https://colab.research.google.com/github/sethkipsangmutuba/Database-Management-System/blob/main/Week_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2 — Indexing and Access Methods

## 1️⃣ Introduction
Efficient data retrieval is central to database performance. As datasets grow, scanning every record for every query becomes impractical. **Indexing** accelerates query processing by letting the DBMS locate records without scanning the entire table.

This week covers **indexing data structures**, trade-offs, and their impact on workload performance.

---

## 2️⃣ Purpose of Indexing
Indexes are **data access structures** used to:
- Reduce query response time for lookups, range queries, and joins.
- Support ordered results without extra sorting.
- Enforce constraints (e.g., uniqueness).
- Enable selective access for large datasets.

Think of a **book index** — jump to relevant pages without scanning all content.

---

## 3️⃣ Design Considerations & Trade-Offs
- **Storage Overhead** — Extra space for index entries.
- **Maintenance Cost** — Inserts, updates, deletes require index updates.
- **Workload Sensitivity** — Read-heavy workloads benefit; write-heavy may suffer.
- **Selectivity** — Best when predicates reduce result set significantly.

---

## 4️⃣ Categories of Indexes

### Primary vs. Secondary
- **Primary** — Built on primary key; table ordered by index.
- **Secondary** — Built on other attributes; independent ordering.

### Clustered vs. Unclustered
- **Clustered** — Dictates physical row order.
- **Unclustered** — Index entries point to separate data storage.

### Dense vs. Sparse
- **Dense** — Entry for every record.
- **Sparse** — Entries for some records; scans between entries.

---

## 5️⃣ B+ Tree Indexes

### 5.1 Overview
Most widely used for **on-disk data**; balanced, supports equality & range queries.

### 5.2 Structure
- **Internal Nodes** — Keys + child pointers.
- **Leaf Nodes** — All index entries + data pointers (or data in clustered form).
- Leaves linked sequentially for fast range scans.

### 5.3 Properties
- Balanced (all leaves at same depth).
- Order-preserving.
- High fan-out → shallow trees → fewer I/Os.

### 5.4 Operations
- **Search** — Traverse from root to leaf.
- **Insertion** — Add entry; split if full.
- **Deletion** — Remove; merge/redistribute if underflow.

### 5.5 Advantages
- Fast range queries.
- Predictable `O(log n)` search.
- Disk-friendly due to large node size.

---

## 6️⃣ Hash Indexes

### 6.1 Concept
Map keys to buckets via a **hash function** for `O(1)` expected lookup on equality.

### 6.2 Structure
- **Buckets** — Store records with same hash.
- **Hash Function** — Maps key → bucket number.
- **Collision Handling** — Chaining or open addressing.

### 6.3 Operations
- Search → hash key → find bucket → scan bucket.
- Insert → hash → place in bucket.
- Delete → remove from bucket.

### 6.4 Pros & Cons
- **Pros** — Very fast equality lookups.
- **Cons** — No natural range query support; depends on hash uniformity.

---

## 7️⃣ Bitmap Indexes

### 7.1 Overview
Use bit vectors to indicate value presence; great for **low-cardinality columns**.

### 7.2 Structure
- One bit vector per distinct value.
- Bit position = row position.
- `1` → value present; `0` → absent.

### 7.3 Operations
- Boolean ops (AND, OR, NOT) combine conditions efficiently.

### 7.4 Strengths & Weaknesses
- **Strengths** — Fast predicate evaluation; great for analytics.
- **Weaknesses** — Poor for high-cardinality; costly to update.

---

## 8️⃣ Advanced Indexing Techniques

### Adaptive Indexing
- Builds/refines indexes as queries run.
- Examples: Database cracking, adaptive merging.

### Composite Indexes
- Multi-column indexes; order matters for query patterns.

### Partial Indexes
- Built on a subset of rows; saves space & maintenance cost.

---

## 9️⃣ Cost Analysis of Indexing
- **Access Time** — Lookup speed.
- **Maintenance Time** — Update overhead.
- **Storage Cost** — Space for index + metadata.
- **Selectivity** — Higher selectivity = more benefit.

---

## 🔟 Choosing the Right Index
Depends on:
- Query workload (range vs equality).
- Data distribution.
- Update frequency.
- Concurrency requirements.

---

## 1️⃣1️⃣ Integration with Query Optimization
Query optimizers use statistics to decide if an index should be used. Sometimes a sequential scan is chosen if the predicate isn’t selective enough.

---

## 📚 1️⃣2️⃣ Reading Assignment
**Primary Reference:**  
Elmasri & Navathe — *Fundamentals of Database Systems*, Ch. 18–19 (indexing).

Focus:
- B+ Tree structure & ops.
- Hash index design.
- Bitmap indexing.
- Index type trade-offs.
- Composite & partial indexing.

---

##  1️⃣3️⃣ Summary
You should be able to:
- Explain purpose & trade-offs of indexing.
- Differentiate primary, secondary, clustered, unclustered indexes.
- Describe B+ Tree, hash, and bitmap indexes.
- Understand adaptive, composite, and partial indexes.
- Evaluate indexing strategies based on workload & cost.
- Appreciate indexing’s role in query optimization.


In [3]:
"""
INDEXING TOOLKIT - Week 2: Indexing and Access Methods

Implements:
1. B+ Tree Index
2. Hash Index
3. Bitmap Index
4. Composite Index
5. Partial Index

Covers:
- Search and update performance trade-offs
- B+Trees, hash indexes, bitmap indexes
- Composite indexes (multi-column)
- Partial indexes (predicate-based)
- Index tuning on an e-commerce dataset

NOTE: Educational implementation — simplified for clarity.
"""

import time                   # For measuring performance
import random                 # For generating random dataset values
import string                 # For random string generation
from collections import defaultdict  # For bitmap index bitmaps


# ==============================
# 1. B+ TREE IMPLEMENTATION
# ==============================

class BPlusTreeNode:
    """A node in a B+ Tree."""
    def __init__(self, is_leaf=False):
        self.is_leaf = is_leaf     # True if this node is a leaf node
        self.keys = []             # Sorted list of keys in this node
        self.children = []         # Pointers to child nodes or actual values


class BPlusTree:
    """Simplified B+ Tree implementation."""
    def __init__(self, order=3):
        self.root = BPlusTreeNode(is_leaf=True)  # Start with an empty leaf node
        self.order = order                       # Max number of keys per node

    def search(self, key, node=None):
        """Search for a key in the tree."""
        if node is None:
            node = self.root
        if node.is_leaf:  # If at leaf, check for exact match
            for i, item in enumerate(node.keys):
                if item == key:
                    return node.children[i]  # Return associated value
            return None  # Not found
        else:
            # Find the correct child pointer to follow
            for i, item in enumerate(node.keys):
                if key < item:
                    return self.search(key, node.children[i])
            return self.search(key, node.children[-1])

    def insert(self, key, value):
        """Insert key-value into the tree."""
        root = self.root
        # If root is full, split it before insertion
        if len(root.keys) == self.order:
            new_root = BPlusTreeNode()
            new_root.children.append(self.root)
            self.split_child(new_root, 0)
            self.root = new_root
        self._insert_non_full(self.root, key, value)

    def _insert_non_full(self, node, key, value):
        """Insert into a non-full node."""
        if node.is_leaf:
            if key not in node.keys:
                # Insert key in sorted order
                idx = 0
                while idx < len(node.keys) and node.keys[idx] < key:
                    idx += 1
                node.keys.insert(idx, key)
                node.children.insert(idx, value)
        else:
            # Find the correct child to insert into
            idx = len(node.keys) - 1
            while idx >= 0 and key < node.keys[idx]:
                idx -= 1
            idx += 1
            if len(node.children[idx].keys) == self.order:
                self.split_child(node, idx)
                if key > node.keys[idx]:
                    idx += 1
            self._insert_non_full(node.children[idx], key, value)

    def split_child(self, parent, index):
        """Split a full child into two nodes."""
        node_to_split = parent.children[index]
        mid = self.order // 2
        new_node = BPlusTreeNode(is_leaf=node_to_split.is_leaf)
        # Move half the keys to new node
        new_node.keys = node_to_split.keys[mid:]
        new_node.children = node_to_split.children[mid:]
        node_to_split.keys = node_to_split.keys[:mid]
        node_to_split.children = node_to_split.children[:mid]
        # Insert new node into parent
        parent.keys.insert(index, new_node.keys[0])
        parent.children.insert(index + 1, new_node)


# ==============================
# 2. HASH INDEX
# ==============================

class HashIndex:
    """Simple hash index using Python's dictionary."""
    def __init__(self):
        self.index = {}  # Key-value pairs

    def insert(self, key, value):
        self.index[key] = value  # O(1) average insertion

    def search(self, key):
        return self.index.get(key, None)  # O(1) average search

    def delete(self, key):
        if key in self.index:
            del self.index[key]  # O(1) average deletion


# ==============================
# 3. BITMAP INDEX
# ==============================

class BitmapIndex:
    """Bitmap index for low-cardinality attributes."""
    def __init__(self):
        self.bitmaps = defaultdict(list)  # Dict of value → bit vector
        self.keys = []                    # Row IDs

    def build(self, data, attribute_index):
        """Build a bitmap index for a given attribute position in the dataset."""
        self.keys = list(range(len(data)))  # Row IDs
        # Identify unique values for the attribute
        distinct_values = sorted(set(row[attribute_index] for row in data))
        # For each distinct value, build a bit vector
        for value in distinct_values:
            self.bitmaps[value] = [
                1 if row[attribute_index] == value else 0 for row in data
            ]

    def search(self, value):
        """Return list of row IDs where the value occurs."""
        return [i for i, bit in enumerate(self.bitmaps[value]) if bit == 1]


# ==============================
# 4. COMPOSITE INDEX
# ==============================

class CompositeIndex:
    """Index on multiple attributes (tuple key)."""
    def __init__(self):
        self.index = {}

    def insert(self, key_tuple, value):
        self.index[key_tuple] = value

    def search(self, key_tuple):
        return self.index.get(key_tuple, None)


# ==============================
# 5. PARTIAL INDEX
# ==============================

class PartialIndex:
    """Index only rows that satisfy a given predicate."""
    def __init__(self, predicate):
        self.index = {}
        self.predicate = predicate  # Function to filter rows

    def insert(self, key, value):
        if self.predicate(value):   # Insert only if predicate is True
            self.index[key] = value

    def search(self, key):
        return self.index.get(key, None)


# ==============================
# 6. PERFORMANCE TESTING
# ==============================

# Generate synthetic e-commerce dataset: (ProductID, Category, Price, Stock)
categories = ["Electronics", "Books", "Clothing", "Sports"]
dataset = [
    (f"P{i:05d}",                        # ProductID
     random.choice(categories),          # Category
     random.randint(5, 500),             # Price
     random.randint(0, 100))              # Stock quantity
    for i in range(10000)
]

# Create index instances
bptree = BPlusTree(order=4)
hash_index = HashIndex()
bitmap_index = BitmapIndex()
bitmap_index.build(dataset, attribute_index=1)  # Build bitmap on 'Category'
composite_index = CompositeIndex()
partial_index = PartialIndex(lambda record: record[2] > 100)  # Price > 100

# Insert dataset into indexes
for record in dataset:
    pid, category, price, stock = record
    bptree.insert(pid, record)
    hash_index.insert(pid, record)
    composite_index.insert((category, price), record)
    partial_index.insert(pid, record)

# Prepare search keys
search_keys = random.sample([row[0] for row in dataset], 2000)

# --- Search without index (linear scan) ---
start = time.time()
for key in search_keys:
    for rec in dataset:
        if rec[0] == key:
            break
linear_time = time.time() - start

# --- Search with B+ Tree ---
start = time.time()
for key in search_keys:
    _ = bptree.search(key)
bptree_time = time.time() - start

# --- Search with Hash Index ---
start = time.time()
for key in search_keys:
    _ = hash_index.search(key)
hash_time = time.time() - start

# --- Search with Bitmap Index ---
category_lookup = dataset[0][1]  # Category of first product
start = time.time()
_ = bitmap_index.search(category_lookup)
bitmap_time = time.time() - start

# ==============================
# 7. OUTPUT RESULTS
# ==============================
print("=== Search Performance (seconds) ===")
print(f"No index (linear scan): {linear_time:.6f}")
print(f"B+ Tree index:         {bptree_time:.6f}")
print(f"Hash index:            {hash_time:.6f}")
print(f"Bitmap index:          {bitmap_time:.6f}")

print("\n=== Example Composite Index Search ===")
sample_category, sample_price = dataset[500][1], dataset[500][2]
print(f"Search for ({sample_category}, {sample_price}):",
      composite_index.search((sample_category, sample_price)))

print("\n=== Example Partial Index Search (Price > 100) ===")
pid_check = dataset[1000][0]
print(f"Product {pid_check} in partial index?:", partial_index.search(pid_check))


=== Search Performance (seconds) ===
No index (linear scan): 0.905849
B+ Tree index:         0.009305
Hash index:            0.001482
Bitmap index:          0.000731

=== Example Composite Index Search ===
Search for (Sports, 304): ('P09278', 'Sports', 304, 43)

=== Example Partial Index Search (Price > 100) ===
Product P01000 in partial index?: ('P01000', 'Electronics', 359, 93)


# Index Performance & Trade-Offs

**No index (linear scan)** → 0.905 seconds — This is the baseline. Every search key forces the DBMS to check each row in the dataset sequentially. Time grows linearly with table size. With 10,000 rows and 2,000 lookups, the cost is noticeable. This is why indexes exist — to avoid touching every row.

**B+ Tree index** → 0.0093 seconds — Dramatic improvement. A B+ Tree reduces search time to O(log n). Here, it’s about 97× faster than a full scan. Suitable for range queries and ordered retrievals.

**Hash index** → 0.00148 seconds — Even faster — about 610× faster than linear scan. Hash lookups are O(1) on average, so performance is constant regardless of table size. Best for exact-match queries but not good for range queries.

**Bitmap index** → 0.00073 seconds — Fastest here — about 1240× faster than full scan. Ideal for low-cardinality attributes (like “Category”). Works by bitwise operations that are extremely fast in memory.

The composite index search result — Searching for `(Sports, 304)` instantly returns `('P09278', 'Sports', 304, 43)`. This shows how a multi-column index can efficiently support queries filtering on multiple fields at once, avoiding two separate lookups.

The partial index search result — Checking if product `P01000` is in the partial index returns `('P01000', 'Electronics', 359, 93)`. The partial index only includes products with Price > 100, so Product P01000 qualifies and is found quickly. Partial indexes save space and speed up lookups when queries often target a subset of rows.

**What this demonstrates for Week 2** — Trade-offs are real. While hash and bitmap indexes are fastest here, they have limitations: hash indexes cannot do range queries; bitmap indexes are only efficient for low-cardinality fields; B+ Trees are slightly slower but handle ranges and ordering well. Index choice depends on workload: for exact product lookups use hash; for category filters use bitmap; for price range queries use B+ Tree. Proper index tuning can change query performance by factors of 100–1000×.
